# MovieLens 100K: Exploratory Data Analysis

This notebook explores the MovieLens 100K dataset to understand its structure, rating distributions, user and movie activity, and data quality before implementing recommendation models.

## 1. Dataset Overview

### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import GENRE_COLS, load_movielens

sns.set_theme(style="whitegrid")

### Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

### Verify Dataset Dimensions

In [ ]:
for name, df in zip(["Ratings", "Movies", "Users"], [ratings, movies, users]):
    print(f"{name}: {df.shape}")

### Examine Unique Users and Movies

In [ ]:
# We wish to determine the number of distinct users and movies in the ratings data.
print("Unique users:", ratings["user_id"].nunique())
print("Unique movies:", ratings["movie_id"].nunique())
print("Total ratings:", len(ratings))

## 2. Rating Distribution

### Rating Frequencies

In [ ]:
freq = ratings["rating"].value_counts().sort_index()
percentage = (freq / len(ratings)) * 100

rating_distribution = pd.DataFrame({
    "Frequency": freq,
    "Percentage (%)": percentage
})

display(rating_distribution)

### Descriptive Statistics

In [ ]:
display(ratings["rating"].describe())

print("Median rating:", ratings["rating"].median())
print("Mode rating:", ratings["rating"].mode().tolist())

### Visualize Rating Distribution

In [ ]:
# Visualization of Rating Distribution
# This shows how frequently users assign each rating
plt.figure(figsize=(8, 5))
sns.barplot(x=freq.index, y=freq.values)

plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.xticks(range(5), [1, 2, 3, 4, 5])

plt.show()

## 3. Movie Activity Analysis

### Ratings per Movie

In [ ]:
movie_activity = ratings.groupby("movie_id").size()

display(movie_activity.describe())

### Most Frequently Rated Movies

In [ ]:
top_movies = (
    movie_activity
    .nlargest(10)
    .rename("rating_count")
    .reset_index()
    .merge(
        movies[["movie_id", "title"]],
        on="movie_id",
        how="left"
    )
    .sort_values("rating_count", ascending=False)
)

display(top_movies[["movie_id", "title", "rating_count"]])

### Movie Activity Distribution

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(movie_activity, bins=30)

plt.title("Distribution of Ratings per Movie")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Movies")

plt.tight_layout()
plt.show()

## 4. User Activity Analysis

### Ratings per User

In [ ]:
user_activity = ratings.groupby("user_id").size()

display(user_activity.describe())

### Most Active Users

In [ ]:
top_users = (
    user_activity
    .nlargest(10)
    .rename("rating_count")
    .reset_index()
)

display(top_users)

### User Activity Distribution

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(user_activity, bins=30)

plt.title("Distribution of Ratings per User")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

## 5. Data Quality Assessment

### Missing Values

In [ ]:
print("Missing values in ratings:")
display(ratings.isnull().sum())

print("Missing values in movies:")
display(movies.isnull().sum())

print("Missing values in users:")
display(users.isnull().sum())

### Duplicate Records

In [ ]:
print("Duplicate rating records:", ratings.duplicated().sum())
print("Duplicate movie records:", movies.duplicated().sum())
print("Duplicate user records:", users.duplicated().sum())

print(
    "Duplicate user-movie pairs:",
    ratings.duplicated(subset=["user_id", "movie_id"]).sum()
)

dup_titles = movies[movies["title"].duplicated(keep=False)].sort_values("title")
print("Movies sharing a duplicate title:", len(dup_titles))
print("Duplicate titles:", dup_titles["title"].nunique())
display(dup_titles[["movie_id", "title"]].head(6))

### Invalid Ratings

In [ ]:
invalid_ratings = ratings[
    ~ratings["rating"].between(1, 5)
]

print("Invalid ratings:", len(invalid_ratings))
display(invalid_ratings.head())

### Dataset Consistency

In [ ]:
missing_users = set(ratings["user_id"]) - set(users["user_id"])
missing_movies = set(ratings["movie_id"]) - set(movies["movie_id"])

print("User IDs without metadata:", len(missing_users))
print("Movie IDs without metadata:", len(missing_movies))

## 6. Genre and Sparsity Analysis

### Matrix Sparsity and Time Span

In [ ]:
n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)

print(f"Matrix sparsity: {sparsity * 100:.2f}% ({len(ratings):,} of {n_users * n_movies:,} pairs rated)")

dates = pd.to_datetime(ratings["timestamp"], unit="s")
print(f"Earliest rating: {dates.min()}")
print(f"Latest rating:   {dates.max()}")
print(f"Span in days:    {(dates.max() - dates.min()).days}")

### Positive Rating Split (rating >= 4)

In [ ]:
positive = ratings["rating"] >= 4
print(f"Positive ratings (>= 4): {positive.sum():,} ({positive.mean() * 100:.2f}%)")
print(f"Ratings below 4:         {(~positive).sum():,} ({(~positive).mean() * 100:.2f}%)")

pos_per_user = ratings.assign(positive=positive).groupby("user_id")["positive"].sum()
display(pos_per_user.describe())
print("Users with 0 positive ratings:", int((pos_per_user == 0).sum()))

### Genre Distribution

In [ ]:
genre_flags = movies[GENRE_COLS]
print("Unique genre flag values:", sorted(pd.unique(genre_flags.to_numpy().ravel()).tolist()))
print("Movies with no genres listed:", int((genre_flags.sum(axis=1) == 0).sum()))
display(movies[movies["unknown"] == 1][["movie_id", "title"]])
display(genre_flags.sum(axis=1).describe())

genre_counts = genre_flags.sum().sort_values(ascending=True)
plt.figure(figsize=(8, 6))
genre_counts.plot(kind="barh")
plt.title("Movies per Genre")
plt.xlabel("Number of Movies")
plt.tight_layout()
plt.show()

### Minimum Ratings Threshold Coverage

In [ ]:
rows = []
for min_r in [1, 10, 25, 50, 100, 200]:
    kept = movie_activity[movie_activity >= min_r]
    rows.append({
        "min_ratings": min_r,
        "movies_kept": len(kept),
        "catalog_pct": 100 * len(kept) / len(movies),
        "ratings_pct": 100 * kept.sum() / len(ratings),
    })

min_ratings_df = pd.DataFrame(rows)
display(min_ratings_df.round(2))

plt.figure(figsize=(8, 4))
plt.bar(min_ratings_df["min_ratings"].astype(str), min_ratings_df["movies_kept"])
plt.title("Movies Qualifying by min_ratings Threshold")
plt.xlabel("min_ratings")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

## 7. Key Findings

### Dataset Overview

[Document the verified number of users, movies, and ratings.]

### Rating Distribution

[Identify the most frequent rating, average rating, and general rating tendencies.]

### Movie Activity

[Identify the most frequently rated movies and describe the distribution of ratings across movies.]

### User Activity

[Describe the distribution of user rating activity and identify any notable patterns.]

### Data Quality

[Summarize missing values, duplicate records, invalid ratings, and inconsistencies.]

### Conclusion

[Summarize the most important observations from the exploratory analysis.]